# N1 · Meet the NHIS

*R for Survey Data, lesson 1 of 7 · about 30 to 40 minutes*

**In this notebook you will learn to:**

- Describe what the NHIS is and why its data need special care
- Load the NHIS teaching file
- Read a codebook entry: the question, who was asked, and what each code means
- Spot special codes such as 7 = Refused and 9 = Don't know

**Before this notebook:** the six R Foundations notebooks, ending with [F6: Combining Tables](https://colab.research.google.com/github/icohr-lab/tutorials/blob/main/notebooks/F6_Combining_Tables.ipynb).

> **Before you start:** click **File > Save a copy in Drive** and work in your copy. Run a code cell by clicking the play button to its left, or press **Shift + Enter**. Go in order from top to bottom.

## 1. The survey

The **National Health Interview Survey (NHIS)** is run every year by the National Center for Health Statistics (NCHS), part of the U.S. Centers for Disease Control and Prevention. Interviewers contact tens of thousands of households across the country and ask about health, health care, and living conditions. In each household, one adult is chosen at random to answer a long set of questions. That person is called the **sample adult**.

NCHS publishes the answers as a **public-use file**: anyone can download it, and nothing in it identifies a person. These lessons use the 2025 Sample Adult file, which has **24,215 adults**.

### The project

Over the next lessons you will answer one research question:

> **Among adults who have had cancer, how does self-rated health differ by income and education?**

### Why survey data need special care

Not every adult in the country had the same chance of ending up in the file. For example, only one adult is interviewed per household, so a person who lives alone is more likely to be chosen than one who lives with three other adults. Some groups are also less likely to respond. NCHS corrects for all of this by giving every person a **weight**: the number of U.S. adults that person stands for. Analyses that ignore the weights describe the 24,215 people in the file, not the country. Lesson N4 is all about this. For now, just know that three columns in the file (`WTFA_A`, `PSTRAT`, `PPSU`) carry this information and must never be dropped.

## 2. Load the data

Run the setup cell. The lines printed by `library(tidyverse)` and `read_csv()` are normal.

In [ ]:
library(tidyverse)

nhis <- read_csv("https://raw.githubusercontent.com/icohr-lab/tutorials/refs/heads/main/data/nhis/nhis2025_adult.csv")

The full NCHS file has 577 columns. This teaching file keeps 16 of them, and **every one of the 24,215 adults**.

> **If you download the full file from NCHS yourself:** add `guess_max = 24215` inside `read_csv()`. By default, `read_csv()` looks at only the first 1,000 rows to decide each column's type, and for questions that few people were asked, that can go wrong. NCHS's own instructions recommend this.

In [ ]:
dim(nhis)

In [ ]:
head(nhis)

## 3. Every answer is a number

Look at the first rows again: apart from `HHX`, every column holds numbers, even for questions like "What is your sex?". Each number is a **code** that stands for an answer. To know what the codes mean, you need the **codebook**.

| Column | What it is |
|---|---|
| `HHX` | Household ID |
| `WTFA_A`, `PSTRAT`, `PPSU` | Survey design: weight, stratum, and sampling unit (lesson N4) |
| `PHSTAT_A` | Self-rated health |
| `CANEV_A` | Ever told by a doctor that you had cancer |
| `NUMCAN_A` | Number of different types of cancer reported |
| `SKNNMCAN_A` | Non-melanoma skin cancer was one of them |
| `POVRATTC_A` | Family income compared with the poverty line |
| `EDUCP_A` | Highest level of education |
| `NOTCOV_A` | Health insurance coverage |
| `AGEP_A` | Age |
| `SEX_A` | Sex |
| `HISPALLP_A` | Race and Hispanic origin |
| `REGION` | Region of the country |
| `URBRRL23` | How urban or rural the county is |

## 4. Reading the codebook

The [2025 Sample Adult codebook](https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Dataset_Documentation/NHIS/2025/adult-codebook.pdf) is a 577-page PDF with one page per variable. Open it in a new tab and search for a variable name (Ctrl+F, or Cmd+F on a Mac). Here is the entry for `PHSTAT_A`, shortened:

**Variable:** PHSTAT_A
**Question Text:** Would you say your health in general is excellent, very good, good, fair, or poor?
**Universe Description:** Sample adults 18+

| Code | Description | Frequency |
|---|---|---|
| 1 | Excellent | 4,789 |
| 2 | Very Good | 8,215 |
| 3 | Good | 7,389 |
| 4 | Fair | 2,970 |
| 5 | Poor | 845 |
| 7 | Refused | 4 |
| 8 | Not Ascertained | 0 |
| 9 | Don't Know | 3 |

Three parts of every entry matter:

- **Question Text**: exactly what was asked.
- **Universe**: who was asked. "Sample adults 18+" means everyone in the file.
- **Codes and frequencies**: what each number means, and how many people gave each answer.

The frequencies are your first check. Count the column yourself and compare.

In [ ]:
nhis %>% count(PHSTAT_A)

The numbers match the codebook exactly, so the file was read correctly. Make this comparison a habit with every new variable.

**Your turn 1.** Count `CANEV_A` and compare your counts with its codebook entry, shown here:

| Code | Description | Frequency |
|---|---|---|
| 1 | Yes | 3,378 |
| 2 | No | 20,804 |
| 7 | Refused | 25 |
| 8 | Not Ascertained | 0 |
| 9 | Don't Know | 8 |

*Expected result: 1 = 3,378, 2 = 20,804, 7 = 25, 9 = 8, matching the codebook (code 8 does not appear because nobody has it)*

In [ ]:
# Your code here

## 5. Special codes

Most NHIS questions use the same codes for answers that are not real answers:

| Code | Meaning |
|---|---|
| 7 | Refused |
| 8 | Not ascertained (the answer was not recorded) |
| 9 | Don't know |

Questions with many categories use two digits instead: **97, 98, 99**. These codes are numbers like any other, so R treats them as real values unless you tell it otherwise.

In [ ]:
nhis %>% count(SEX_A)

1 is male and 2 is female, but 25 people have 7 or 9. If you calculated "percent female" as the share of all 24,215 adults with code 2, those 25 would quietly count as "not female".

Age is a sharper example. Look at the oldest ages in the file.

In [ ]:
nhis %>% filter(AGEP_A >= 85) %>% count(AGEP_A)

Two different things are going on:

- **85 means "85 or older".** To protect privacy, NCHS does not publish exact ages above 84. This is called **top-coding**. The 963 people coded 85 are not all 85 years old.
- **97 and 99 are not ages at all.** They are "Refused" and "Don't know". Nobody in the file is 97 or 99.

Codes are labels, not quantities. Before you calculate anything, find out which codes are real answers. Lesson N2 turns the special codes into missing values (`NA`).

**Your turn 2.** How many adults have an age code that is not an age? Use `sum()` and `%in%`.

*Expected result: 60 (56 refused, 4 don't know)*

In [ ]:
# Your code here

## 6. Who was asked: the universe

Not every question is asked of everyone. `SKNNMCAN_A` records whether non-melanoma skin cancer was one of a person's cancers. Its codebook universe is:

> Sample adults 18+ **who were ever told they had cancer**

People outside the universe were never asked, so their value is blank in the file, which R reads as `NA`.

In [ ]:
nhis %>% count(SKNNMCAN_A)

**Your turn 3.** 20,837 people have `NA` for `SKNNMCAN_A`. Show where that number comes from by counting `CANEV_A` and `SKNNMCAN_A` together: `count(CANEV_A, SKNNMCAN_A)`.

*Expected result: every `NA` belongs to someone who did not answer Yes to `CANEV_A`: 20,804 No + 25 Refused + 8 Don't know = 20,837*

In [ ]:
# Your code here

## 7. A number that really is a number: income

`POVRATTC_A` compares a family's income with the federal **poverty threshold** for a family of that size. A value of 1 means income exactly at the poverty line; 2 means twice the poverty line; 0.5 means half of it.

In [ ]:
summary(nhis$POVRATTC_A)

Two things to know about this column:

- **It is top-coded at 12.** Families with income 12 or more times the poverty line all appear as 12, again to protect privacy.
- **Some values are imputed.** About one in five adults did not report an exact family income (some gave only a range, some nothing). For them, NCHS filled in a statistically estimated value based on similar families, a process called **imputation**. The result has no missing values. These lessons use the column as NCHS provides it; studies that depend heavily on income sometimes use NCHS's more advanced multiple-imputation files instead.

**Your turn 4.** How many adults have family income **below** the poverty line? And how many are top-coded at 12?

*Expected result: 2,211 below the poverty line; 1,258 top-coded at 12*

In [ ]:
# Your code here

## 8. Your turn: education

Here are the codes for `EDUCP_A` from the codebook:

| Code | Description |
|---|---|
| 1 | Grade 0 to 11 |
| 2 | 12th grade, no diploma |
| 3 | GED or equivalent |
| 4 | High school graduate |
| 5 | Some college, no degree |
| 6 | Associate degree: occupational, technical, or vocational program |
| 7 | Associate degree: academic program |
| 8 | Bachelor's degree |
| 9 | Master's degree |
| 10 | Professional school or doctoral degree |
| 97 | Refused |
| 99 | Don't know |

**Your turn 5.** Count `EDUCP_A`. Then answer two questions with `sum()` and `%in%`: how many adults have a bachelor's degree or higher, and how many refused or did not know?

*Expected result: 9,577 with a bachelor's degree or higher; 83 refused or did not know (28 + 55)*

In [ ]:
# Your code here

---
## Recap

| Code | What it does |
|---|---|
| `nhis %>% count(x)` | Count each code; compare with the codebook |
| `nhis %>% count(x, y)` | Count combinations of two columns |
| `sum(x %in% c(7, 8, 9))` | How many special codes |
| `8:10` | The numbers 8, 9, 10 |
| `summary(x)` | Minimum, quartiles, mean, and maximum of a number column |
| `read_csv(file, guess_max = 24215)` | Read a full NCHS file correctly |

---
## Solutions

Try each exercise before looking. There is often more than one correct way to write it.

**Your turn 1**

```r
nhis %>% count(CANEV_A)
```

**Your turn 2**

```r
sum(nhis$AGEP_A %in% c(97, 98, 99))
```

**Your turn 3**

A missing value can mean "not asked" as well as "no answer". The universe tells you which.

```r
nhis %>% count(CANEV_A, SKNNMCAN_A)
```

**Your turn 4**

```r
sum(nhis$POVRATTC_A < 1)
sum(nhis$POVRATTC_A == 12)
```

**Your turn 5**

`8:10` is a shortcut for `c(8, 9, 10)`.

```r
nhis %>% count(EDUCP_A)
sum(nhis$EDUCP_A %in% 8:10)
sum(nhis$EDUCP_A %in% c(97, 98, 99))
```

---
## Next

Continue with [N2: Preparing Variables](https://colab.research.google.com/github/icohr-lab/tutorials/blob/main/notebooks/N2_Preparing_Variables.ipynb).